# **Part 1: Building simple Convolutional Neural Networks using Pytorch for MNIST dataset**


**Step 1: Import Required Libraries**

In [1]:
print("Kernel is working")

Kernel is working


In [2]:
import sys
print("Python:", sys.version, flush=True)
print("Environment:", sys.executable, flush=True)

print("Loading NumPy...", flush=True)
import numpy as np
print("NumPy loaded:", np.__version__, flush=True)

print("Loading scikit-learn...", flush=True)
from sklearn.model_selection import train_test_split
print("Both imports worked", flush=True)

Python: 3.12.14 (main, Sep  3 2026, 03:46:19) [Clang 21.1.8 ]
Environment: /Users/arthur/.local/share/mamba/envs/cs135_26f_env/bin/python
Loading NumPy...
NumPy loaded: 2.4.6
Loading scikit-learn...
Both imports worked


In [1]:
import torch           # Main PyTorch library - provides tensors (like numpy arrays but GPU-compatible)
import torch.nn as nn  # Neural network building blocks: layers (Linear, Conv2d), loss functions (CrossEntropyLoss)
import torch.optim as optim  # Optimization algorithms that update model weights during training like SGD, Adam
import torch.nn.functional as F  # Stateless functions (no learnable parameters) like activation functions (relu, sigmoid), loss functions, pooling
from torch.utils.data import Dataset, DataLoader, random_split  # DataLoader used since PyTorch datasets are often too big for memory
import torchvision.datasets as datasets  # Built-in benchmark datasets ready to download and use like MNIST, CIFAR-10
import torchvision.transforms as transforms # Image preprocessing and augmentation pipelines such as ToTensor()

import numpy as np
from sklearn.model_selection import train_test_split

# print("PyTorch version:", torch.__version__)

: 

**Step 2: Load Data and setup Data Loaders**

The **MNIST dataset** is a collection of handwritten digits commonly used for training various image processing systems. It includes 60,000 training images and 10,000 testing images, each 28x28 pixel grayscale image of the digits 0 through 9. We will train a CNN to classify the label for a grayscale image.

In [ ]:
batch_size = 64

### LOAD DATA ###

# Download the MNIST training set
# transform=transforms.ToTensor() converts images from a Python Image Library format to a PyTorch tensor
full_train_dataset = datasets.MNIST(root='dataset/', train=True, transform=transforms.ToTensor(), download=True)

# Split the 60,000 training images into:
# train_dataset -> 50,000 images used to model weights
# val_dataset -> 10,000 images to monitor performance on unseen data during training
train_dataset, val_dataset = random_split(full_train_dataset, [50000, 10000])

# Wrap training set in a DataLoader:
# shuffle=True → randomizes image order every epoch so the model doesn't memorize the sequence
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)

# Wrap validation set in a DataLoader:
# shuffle=False → order doesn't matter for evaluation, and keeping it fixed is more reproducible
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)

# Download the MNIST test set (10,000 images held out until final evaluation)
# train=False → loads the test split (10,000 images)
# The test set is only used ONCE at the very end to report final model performance.
test_dataset = datasets.MNIST(root='dataset/', train=False, transform=transforms.ToTensor(), download=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)


Let's visualize what the different classes in the MNIST dataset looks like

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get one image per class (0-9)
class_images = {}
for images, labels in train_loader:
    for img, label in zip(images, labels):
        label = label.item()
        if label not in class_images:
            class_images[label] = img
        if len(class_images) == 10:
            break
    if len(class_images) == 10:
        break

# Plot one example per class
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('MNIST Dataset - One Example Per Class', fontsize=14)

for digit, ax in enumerate(axes.flat):
    img = class_images[digit].squeeze().numpy()
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Class: {digit}')
    ax.axis('off')

plt.tight_layout()
plt.show()

Let's inspect the dimensions of dataset and shape of each image

In [ ]:
# ── DATASET SIZE ─────────────────────────────────────────────────────────────
print(f"Training samples:   {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples:       {len(test_dataset)}")

# ── IMAGE SIZE ───────────────────────────────────────────────────────────────
# Grab a single image and label from the dataset
sample_image, sample_label = full_train_dataset[0]

print(f"\nRaw tensor shape: {sample_image.shape}")   # (C, H, W)
print(f"  Channels: {sample_image.shape[0]}  ← grayscale=1, RGB=3")
print(f"  Height:   {sample_image.shape[1]}  ← pixels")
print(f"  Width:    {sample_image.shape[2]}  ← pixels")
print(f"  Label:    {sample_label}")

# ── HOW THIS MAPS TO THE CNN ──────────────────────────────────────────────────
print(f"\nCNN input settings:")
print(f"  in_channels = {sample_image.shape[0]}  ← matches number of image channels")
print(f"  in_features = {sample_image.shape[0]} × {sample_image.shape[1]} × {sample_image.shape[2]} = {sample_image.shape[0] * sample_image.shape[1] * sample_image.shape[2]}")

**Step 3: Create a simple Convolutional Neural Network (CNN) for image classification**

We have provided code below that implements the following CNN for classifying MNIST:

**Architecture**:
        
        Input (1, 28, 28)

        Conv1 → ReLU         (1, 28, 28)  → (8, 28, 28)

        MaxPool              (8, 28, 28)  → (8, 14, 14)

        Conv2 → ReLU         (8, 14, 14)  → (16, 14, 14)

        MaxPool              (16, 14, 14) → (16, 7, 7)
        
        Flatten              (16, 7, 7)   → (784,)

        Fully Connected      (784,)       → (10,)

        Output: 10 raw scores (one per digit class)

In [ ]:
class CNN(nn.Module):
    """
      nn.Module is the base class for all PyTorch models. By inheriting from it we get built-in functionality for tracking parameters,
      saving/loading weights,and moving the model to GPU with .to(device)
    """
    def __init__(self, in_channels=1, num_classes=10):
        """
          __init__ is where all layers are defined and stored as attributes.
          Layers with learnable parameters (Conv2d, Linear) must be defined here
          so PyTorch can track and update their weights during training.

          Args:
            in_channels (int): Number of input channels (1 for grayscale images like MNIST, 3 for RGB like CIFAR-10)
            num_classes (int): The number of output classes (10 for digit recognition 0-9 in MNIST)
        """
        # Always call the parent class constructor first.
        # This initialises PyTorch's internal bookkeeping for parameters and layers
        super(CNN, self).__init__()

        # CONVOLUTIONAL LAYER 1
        # First convolutional layer with 8 filters (or kernels), which applies 3x3 convolution to input image
        # Output shape: (batch, 8, 28, 28)
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))

        # MAX POOLING LAYER
        # Reduces spatial dimensions by taking the maximum value in each 2×2 window.
        # stride=(2,2) means the window moves 2 pixels at a time → halves H and W
        # Applied after BOTH conv1 and conv2:
        #   After conv1: (8,  28, 28) → (8,  14, 14)
        #   After conv2: (16, 14, 14) → (16,  7,  7)
        self.pool = nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2))

        # CONVOLUTIONAL LAYER 2
        # in_channels=8 must match out_channels of conv1
        # Second convolutional layer with 16 filters, using 3x3 kernels
        # Output shape: (batch, 16, 14, 14)
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))

        # FULLY CONNECTED LAYER
        # After two rounds of conv + pool the feature map is (16, 7, 7).
        # We flatten this into a 1D vector of size 16 × 7 × 7 = 784.
        # This layer maps those 784 features to num_classes (10) output scores,
        # one score per digit class. The highest score is the model's prediction.
        # No activation here — raw scores (logits) are passed to CrossEntropyLoss
        # which applies softmax internally
        self.fc1 = nn.Linear(16 * 7 * 7, num_classes)


    def forward(self, x):
      """
      forward() defines how data flows through the layers defined in __init__.
      PyTorch calls this automatically when you do: output = model(input)
      You should never call forward() directly.

      Args:
            x (tensor): Input batch of images, shape (batch_size, 1, 28, 28)
                        e.g. with batch_size=64: (64, 1, 28, 28)

      Returns:
            x (tensor): Raw class scores (logits), shape (batch_size, 10)
                        These are NOT probabilities yet — pass through softmax
                        to get probabilities, or directly into CrossEntropyLoss

      """
      # BLOCK 1: Conv → ReLU → Pool
      # Apply conv1
      # ReLU sets all negative values to 0, introducing non-linearity so the
      # network can learn complex patterns beyond simple linear transformations.
      # Shape: (batch, 1, 28, 28) → (batch, 8, 28, 28)
      x = F.relu(self.conv1(x))

      # MaxPool halves the spatial dimensions to reduce computation
      # Shape: (batch, 8, 28, 28) → (batch, 8, 14, 14)
      x = self.pool(x)

      # BLOCK 2: Conv → ReLU → Pool
      # Apply conv2
      # Shape: (batch, 8, 14, 14) → (batch, 16, 14, 14)
      x = F.relu(self.conv2(x))

      # MaxPool again halves spatial dimensions
      # Shape: (batch, 16, 14, 14) → (batch, 16, 7, 7)
      x = self.pool(x)

      # FLATTEN
      # Reshape the 3D feature map into a 1D vector per image in the batch.
      # x.shape[0] preserves the batch dimension.
      # -1 tells PyTorch to infer the remaining size: 16 × 7 × 7 = 784
      # Shape: (batch, 16, 7, 7) → (batch, 784)
      x = x.reshape(x.shape[0], -1)

      # FULLY CONNECTED
      # Map the 784 learned features to 10 output scores (one per digit class).
      # The class with the highest score is the model's predicted digit.
      # Shape: (batch, 784) → (batch, 10)
      x = self.fc1(x)

      # Return raw logits — CrossEntropyLoss expects these directly
      return x

In [ ]:
# Training and Validation functions

def train_step(model, data, targets, loss_fn, optimizer):
  """
    Performs one mini-batch update during training:
        1. Forward pass  → compute predictions and loss
        2. Backward pass → compute gradients via backpropagation
        3. Weight update → optimizer adjusts model parameters

    Args:
        model     : The CNN model being trained
        data      : One batch of input images,  shape (batch_size, 1, 28, 28)
        targets   : One batch of true labels,   shape (batch_size,)
        loss_fn   : Loss function e.g. nn.CrossEntropyLoss()
        optimizer : Optimizer e.g. optim.Adam()

    Returns:
        loss.item()  : Scalar loss value for this batch (plain Python float, not a tensor)
        num_correct  : Number of correct predictions in this batch
        num_samples  : Total number of samples in this batch (usually = batch_size)
  """
  model.train() # Set the model to training mode. This enables layers that behave differently during training vs evaluation

  # MOVE DATA TO DEVICE
  # PyTorch requires ALL tensors involved in a computation to be on the SAME device.
  # .to(device) moves the tensor to whichever device was set earlier
  data = data.to(device=device)
  targets = targets.to(device=device)

  # FORWARD PASS
  # Pass the batch of images through the model.
  # Internally calls model.forward(data) and returns raw class scores (logits)
  # Shape: (batch_size, num_classes) e.g. (64, 10) for MNIST
  scores = model(data)

  # Compute how wrong the model's predictions are compared to the true labels.
  # CrossEntropyLoss applies softmax to scores internally, then computes
  # negative log likelihood. Returns a single scalar value averaged over the batch.
  loss = loss_fn(scores, targets)

  # BACKWARD PASS
  optimizer.zero_grad()  # set all gradients to zero for each batch bc by default, gradients are accumulated in PyTorch
  loss.backward() # backpropagation to compute gradient of the loss with respect to all trainable parameters in the model

  # gradient descent
  optimizer.step() # optimizer updates model parameters using gradients computed during backpropagation

  # Accumulate loss and accuracy
  _, predictions = scores.max(1)
  num_correct = (predictions == targets).sum().item()
  num_samples = predictions.size(0)

  return loss.item(), num_correct, num_samples


def validation_step(model, data, targets, loss_fn):
  """
    Performs one mini-batch evaluation during validation.
    No gradient computation or weight updates occur here —
    we are only measuring how well the model generalises to unseen data.

    Args:
        model   : The CNN model being evaluated
        data    : One batch of input images,  shape (batch_size, 1, 28, 28)
        targets : One batch of true labels,   shape (batch_size,)
        loss_fn : Loss function e.g. nn.CrossEntropyLoss()

    Returns:
        loss.item()           : Scalar loss value for this batch
        num_correct           : Number of correct predictions in this batch
        num_samples           : Total samples in this batch
  """
  model.eval() # Set the model to evaluation mode

  # Move batch to the same device as the model
  data = data.to(device=device)
  targets = targets.to(device=device)

  # During validation we never call loss.backward(), so gradients are unnecessary.
  with torch.no_grad(): # Disable gradient calculation for validation

    # Forward pass only — no backward pass or weight update
    scores = model(data)
    loss = loss_fn(scores, targets)

    # Same accuracy calculation as train_step
    _, predictions = scores.max(1)
    num_correct = (predictions == targets).sum().item()
    num_samples = predictions.size(0)

  return loss.item(), num_correct, num_samples



In [ ]:
# DEVICE SETUP
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# HYPERPARAMETERS
in_channel = 1
num_classes = 10 # MNIST has 10 classes (one for each digit from 0 to 9)
learning_rate = 0.001 # step size used for each iteration
batch_size = 64

# INITIALISE NETWORK
model = CNN().to(device)

# LOSS FUNCTION
loss_fn = nn.CrossEntropyLoss()

# OPTIMIZER
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

Train for a certain number of epochs, 5 epochs can take about 1 minute to run

In [ ]:
# -------- TRACKING METRICS ---------
train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

EPOCHS = 5

# -------- TRAINING LOOP -----------
for epoch in range(EPOCHS):
  # Training accumulators
  running_loss = 0.0 # sum of batch losses across the epoch
  num_correct = 0    # total correct predictions across all training batches
  num_samples = 0    # total samples seen across all training batches

  # Validation accumulators — separate from training so metrics don't mix
  val_running_loss = 0.0
  val_num_correct = 0
  val_num_samples = 0

  # TRAINING PHASE: Iterate over every mini-batch in the training set.
  # enumerate() gives us batch_idx (the batch number) and the batch itself
  # (data, targets) where:
  #   data    → image tensors, shape (batch_size, 1, 28, 28)
  #   targets → true digit labels, shape (batch_size,)
  # One full pass through train_loader = one epoch of training
  for batch_idx, (data, targets) in enumerate(train_loader):
    # train_step performs forward pass, loss computation, backpropagation, and weight update for this single batch
    loss, num_corr, num_sam = train_step(model, data, targets, loss_fn, optimizer)
    running_loss += loss
    num_correct += num_corr
    num_samples += num_sam

  # VALIDATION PHASE: After all training batches are done, evaluate on the validation set.
  for batch_idx, (data, targets) in enumerate(val_loader):
    loss, num_corr, num_sam = validation_step(model, data, targets, loss_fn)
    val_running_loss += loss
    val_num_correct += num_corr
    val_num_samples += num_sam

  # EPOCH-LEVEL METRICS
  epoch_loss = running_loss / len(train_loader) # Average loss per batch
  epoch_acc = float(num_correct) / float(num_samples) * 100

  # Append epoch averages to history lists for plotting later
  train_losses.append(epoch_loss)
  train_accuracies.append(epoch_acc)

  # Same averaging for validation metrics
  val_epoch_loss = val_running_loss / len(val_loader)
  val_epoch_acc = float(val_num_correct) / float(val_num_samples) * 100

  val_losses.append(val_epoch_loss)
  val_accuracies.append(val_epoch_acc)

  print(
      f'Epoch {epoch+1}, '
      f'Loss: {epoch_loss:0.2f}, '
      f'Train Acc: {epoch_acc:0.2f}, '
      f'Val Loss: {val_epoch_loss:0.2f}, '
      f'Val Acc: {val_epoch_acc:0.2f}'
  )



In [ ]:
# ---- Plot Training Curves ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('CNN Training on MNIST', fontsize=14)

ax1.plot(range(1, EPOCHS+1), train_losses, marker='o', color='royalblue', label='Train')
ax1.plot(range(1, EPOCHS+1), val_losses,   marker='o', color='orange', label='Validation')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(range(1, EPOCHS+1), train_accuracies, marker='o', color='royalblue', label='Train')
ax2.plot(range(1, EPOCHS+1), val_accuracies,   marker='o', color='orange',    label='Validation')

ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Test the Model
def test_model(loader, model):

    if loader.dataset.train:
        print("Accuracy on training data")
    else:
        print("Accuracy on testing data")

    num_correct = 0
    num_samples = 0
    model.eval()

    with torch.no_grad():  # dont compute gradients
        for x, y in loader:
            x = x.to(device=device)
            y = y.to(device=device)

            scores = model(x)
            _, predictions = scores.max(1)

            num_correct += (predictions == y).sum()
            num_samples += predictions.size(0)

        print(f'Got {num_correct}/{num_samples} with accuracy {float(num_correct)/float(num_samples)*100: .2f}')

    model.train()

test_model(test_loader, model)

# **Problem 1**: Experiment with the CNN model and the MNIST dataset. Please do the following:

**Part a:** Try out different optimizers with the initial hyperparameters found in the starter notebook. Discuss the effects of the different optimizers, which one you think performs the best on MNIST, and why.




In [ ]:
# Implement Problem 1 Part A

**Part b: Batch Size and Learning Rate**
Given the best optimizer you found, experiment with batch size and learning rate.

In [ ]:
# Implement Problem 1 Part B

# **Part 2: T-Cell Image Data Classification**

**Immunotherapy** is a cancer treatment that harnesses the body's own immune cells to attack tumors. T-cells are an important part of the immune system and a common target for immunotherapy. However, for most immunotherapy treatments that target T-cells to be effective, the T-cells used must be in an active state. Being able to quickly and accurately classify T-cells by activation state is therefore critical to the success of many immunotherapy treatments.

Our goal is to classify the grayscale image of a single T-cell as **Active** (capable of attacking cancer cells, selected for treatment) or **Quiecent** (inactive, discarded from the treatment pipeline). This is a **binary image classification** problem.

More information about this dataset can be found [here](https://carpentries-incubator.github.io/ml4bio-workshop/02-T-cells/index.html).

**Step 1: Download T-Cell Image Data**

The data was stored on a Google drive which can be downloaded inside a notebook.

In [ ]:
import gdown
!gdown 1zg0hRua1RWqR2EFwVzChK3sVNKkctNh9
!gdown 1_g_8dLvjLb277snQ-EEDDRvSSVkmnsvN

In [ ]:
X = np.loadtxt("X_train.csv")
y = np.loadtxt("y_train.csv")

X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.1, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.2, random_state=42)

**Step 2: Reshape and normalize the data**

In [ ]:
X_train = X_train/255.0
X_val = X_val/255.0
X_test = X_test/255.0

In [ ]:
print(X_train.shape)

X_train = X_train.reshape((-1, 1, 78, 78)).astype('float32') # Reshape for PyTorch (batch, channels, height, width)
X_val = X_val.reshape((-1, 1, 78, 78)).astype('float32')
X_test = X_test.reshape((-1, 1, 78, 78)).astype('float32')

**Step 3: Create tensorflow datasets/batches**


In [ ]:
class TCellDataset(Dataset):
  def __init__(self, features, labels):
    self.features = torch.tensor(features, dtype=torch.float32)
    self.labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1) # Add channel dimension for labels

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, idx):
    return self.features[idx], self.labels[idx]

train_dataset = TCellDataset(X_train, y_train)
val_dataset = TCellDataset(X_val, y_val)

BATCH_SIZE = 32
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

#**Problem 2: Simple PyTorch CNN Model for T-cell binary image classification**

**Complete the CNN class with the following architeture for the t-cell dataset.** You can follow along the subtasks listed in the incomplete code below.  Make sure to look into the size of the images in the data set to help determine the dimensions of the different layers.


**Architecture:**

Input image (1 channel, 78x78)

Conv Layer 1 -> ReLU -> MaxPool

Conv Layer 2 -> ReLU

Flatten

Fully Connected 1 -> ReLU

Fully Connected 2 -> ReLU

Output (1 neuron) -> Sigmoid




In [ ]:
class SimpleCNN(nn.Module):
  def __init__(self):
    super().__init__()
    # Subtask 1: Create a 2D convolution layer
    #   - Input channels:  1  (grayscale image)
    #   - Output channels: 16 (number of filters to learn)
    #   - Kernel size:     5  (5×5 filter)
    #   - Padding:        'same' (output H and W stay the same as input)
    self.conv1 = # TO DO

    # Subtask 2: Add a ReLU activation
    self.relu1 = # TO DO

    # Subtask 3: Add a MaxPool layer to halve the spatial dimensions
    #   - Kernel size: 2
    #   - Stride:      2
    #   Hint: After this layer a 78×78 image becomes 39×39
    self.maxpool1 = # TO DO

    # Subtask 4: Create a second 2D convolution layer
    #   - Input channels:  16 (must match output of conv1)
    #   - Output channels: 8  (we reduce the number of filters)
    #   - Kernel size:     3
    #   - Padding:        'same'
    self.conv2 = None  # TO DO

    # Subtask 5: Add a ReLU activation
    self.relu2 = None  # TO DO

    # Subtask 6: Add a Flatten layer to convert the 3D feature map to 1D
    # Hint: After conv2 the feature map is (8, 39, 39) so the flattened size is 8 × 39 × 39 = 12168
    self.flatten = None  # TO DO

    # FULLY CONNECTED LAYERS
    # Subtask 7: Add a Linear layer
    #   - Input features:  8 * 39 * 39 = 12168
    #   - Output features: 24
    self.d1 = None  # TO DO

    # Subtask 8: Add a ReLU activation
    self.relu3 = None  # TO DO

    # Subtask 9: Add a Linear layer
    #   - Input features:  24
    #   - Output features: 6
    self.d2 = None  # TO DO

    # Subtask 10: Add a ReLU activation
    self.relu4 = None  # TO DO

    # OUTPUT LAYER
    # Subtask 11: Add the final Linear layer
    #   - Input features:  6
    #   - Output features: 1  (binary classification → single probability)
    self.pred = None  # TO DO

    # Subtask 12: Add a Sigmoid activation to squash output between 0 and 1
    self.sigmoid = None  # TO DO

  def forward(self, x):
    """
        Define the forward pass by connecting layers in order.
        Each task below corresponds to applying one layer to x.

        Hint: the pattern is always  x = self.layer(x)
    """

    # Subtask 13: Pass x through conv1, then relu1, then maxpool1
    # YOUR CODE HERE

    # Subtask 14: Pass x through conv2, then relu2
    # YOUR CODE HERE

    # Subtask 15: Flatten x
    # YOUR CODE HERE

    # Subtask 16: Pass x through d1, then relu3
    # YOUR CODE HERE

    # Subtask 17: Pass x through d2, then relu4
    # YOUR CODE HERE

    # Subtask 18: Return the sigmoid of pred(x)
    # YOUR CODE HERE
    return None

#**Problem 3: T-Cell Image Classification with Binary CNN**

**Part a**: Use the template from the CNN MNIST example to define the loss function (BCE), optimizer and set up a training and validation loop with the simple binary CNN you just developed and apply this to the T-Cell Image data set. Please train your model for at least 15 epochs and graph the training + validation loss and accuracies.
Use the following hyperparameters:

1.   batch size = 32
2.   loss function = BCELoss
3.   optimizer = Adam optimizer
4.   learning rate = 0.001


In [ ]:
# Problem 3A

**Part b**:
Experiment with the architecture of the CNN model. You may but are not required to also vary any hyperparameters you desire. You do not need to by systematic explore a set of networks, alter the layers and try to get a feel for what works best in this scenario. However, **only calculate test accuracy after you choose a final architecture**. Report what you found to work best, training and validation loss, and the testing AUCROC of the final model you choose.

In [ ]:
# Problem 3B

**Part C:** Discuss what you found. What neural network architecture changes did you try? What seemed to work well and what didn't work well in this application? Why do you think so?

### 3C Answer